In [0]:
CATALOG = "hindsight_dev"
MAP = f"{CATALOG}.silver.concept_map"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {MAP} (
    src_tag        STRING  COMMENT 'Tag as it appears in num.txt',
    canonical_tag  STRING  COMMENT 'Concept it rolls up to',
    statement      STRING  COMMENT 'BS | IS | CF | EPS',
    mapping_method STRING  COMMENT 'SELF | TAXONOMY_MIGRATION | MANUAL',
    confidence     STRING  COMMENT 'HIGH | MEDIUM | LOW',
    note           STRING
)
USING DELTA
COMMENT 'Source tag -> canonical concept. Managed as data so scope can change without a deploy.'
""")

In [0]:
DECISIONS = [
    # --- Revenue: ASC 606 split the concept in 2018 ---
    ("Revenues",                                          "REVENUE", "IS", "TAXONOMY_MIGRATION", "HIGH",   "Generic top line, spans both regimes"),
    ("RevenueFromContractWithCustomerExcludingAssessedTax","REVENUE", "IS", "TAXONOMY_MIGRATION", "HIGH",   "Post-ASC 606 standard tag"),
    ("SalesRevenueNet",                                   "REVENUE", "IS", "TAXONOMY_MIGRATION", "HIGH",   "Pre-ASC 606 standard tag"),
    ("SalesRevenueGoodsNet",                              "REVENUE", "IS", "TAXONOMY_MIGRATION", "MEDIUM", "Pre-606, goods only; understates services cos"),

    # --- Cost of revenue: NOT CostsAndExpenses ---
    ("CostOfRevenue",              "COST_OF_REVENUE", "IS", "TAXONOMY_MIGRATION", "HIGH",   ""),
    ("CostOfGoodsAndServicesSold", "COST_OF_REVENUE", "IS", "TAXONOMY_MIGRATION", "HIGH",   "Post-2018 preferred tag"),
    ("CostOfGoodsSold",            "COST_OF_REVENUE", "IS", "TAXONOMY_MIGRATION", "MEDIUM", "Goods only"),
    ("CostsAndExpenses",           "TOTAL_COSTS",     "IS", "MANUAL",             "HIGH",   "TOTAL operating costs, NOT cost of revenue. Deliberately separate."),

    # --- Income: parent vs including NCI are different concepts ---
    ("NetIncomeLoss", "NET_INCOME",            "IS", "SELF",   "HIGH", "Attributable to parent"),
    ("ProfitLoss",    "NET_INCOME_INCL_NCI",   "IS", "MANUAL", "HIGH", "Includes noncontrolling interests. NOT the same as NetIncomeLoss."),

    # --- Cash: ASU 2016-18 changed restricted cash treatment ---
    ("CashAndCashEquivalentsAtCarryingValue", "CASH",              "BS", "SELF",   "HIGH", "Excludes restricted cash"),
    ("Cash",                                  "CASH",              "BS", "MANUAL", "MEDIUM", "Legacy narrow tag"),
    ("CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents",
                                              "CASH_INCL_RESTRICTED","BS","MANUAL","HIGH", "Post-ASU 2016-18. Includes restricted. Separate concept."),
]

STATEMENT_HINTS = {  # for the bulk self-maps
    "BS": ["Assets","Liabilities","StockholdersEquity","Goodwill","Inventory",
           "PropertyPlant","AccountsPayable","AccountsReceivable","RetainedEarnings",
           "CommonStock","PreferredStock","AdditionalPaidIn","AccumulatedOther","Deposits"],
    "CF": ["NetCashProvided","PaymentsTo","ProceedsFrom","IncreaseDecreaseIn",
           "Depreciation","ShareBasedCompensation","RepaymentsOf"],
    "EPS":["EarningsPerShare","PerBasicShare","PerDilutedShare","ParOrStatedValue"],
}

In [0]:
from pyspark.sql import functions as F

decided = {d[0] for d in DECISIONS}

# same query you just ran, as a DataFrame
cands = spark.sql("""
WITH c AS (
  SELECT tag, adsh FROM hindsight_dev.bronze.num_raw
  WHERE (segments IS NULL OR trim(segments)='')
    AND (coreg    IS NULL OR trim(coreg)   ='')
    AND uom='USD' AND version LIKE 'us-gaap%' AND value IS NOT NULL
)
SELECT tag, count(DISTINCT adsh) AS filings FROM c GROUP BY tag
HAVING count(DISTINCT adsh) >= 0.05 * (SELECT count(DISTINCT adsh) FROM c)
""").collect()

def guess_stmt(tag):
    for stmt, keys in STATEMENT_HINTS.items():
        if any(k.lower() in tag.lower() for k in keys):
            return stmt
    return "IS"

rows = list(DECISIONS)
for r in cands:
    if r.tag not in decided:
        rows.append((r.tag, r.tag, guess_stmt(r.tag), "SELF", "HIGH", ""))

spark.createDataFrame(rows, "src_tag string, canonical_tag string, statement string, "
                            "mapping_method string, confidence string, note string") \
     .write.mode("overwrite").saveAsTable(MAP)

print(f"{len(rows)} mappings written")
display(spark.sql(f"""
  SELECT canonical_tag, count(*) n_source_tags, collect_list(src_tag) tags
  FROM {MAP} WHERE mapping_method <> 'SELF' GROUP BY canonical_tag ORDER BY n_source_tags DESC
"""))